In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

# load dataset
path = "./../project/Dataset/BOM.csv"
df = pd.read_csv(path)

# drop completely empty rows
df = df.dropna(how='all')

# rename columns to integers
df.columns = range(df.shape[1])

# drop first column if it's name/location
df = df.drop(columns=[0])

# find target column (assume last column)
target_col = df.columns[-1]

# drop rows where target is NaN
df = df[df[target_col].notna()]

# separate target
Y = df.pop(target_col)
Y = Y.map({'No':0, 'Yes':1})

X = df

# separate numeric and categorical columns
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# impute missing values
X[num_cols] = SimpleImputer(strategy='mean').fit_transform(X[num_cols])
X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])

# one-hot encode categorical columns
X = pd.get_dummies(X, drop_first=True)
X.columns = X.columns.astype(str)

# scale features
scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# train-test split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


# try different k for KNN
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
best_k = None
best_score = -float('inf')

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(x_train, y_train)
    y_pred = knn.predict(x_test)
    score = r2_score(y_test, y_pred)
    print(f"k={k}, R² score={score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = k

print("\n best k:", best_k)
print(" best R^2 score:", best_score)

k=1, R² score=-0.3870
k=3, R² score=0.0515
k=5, R² score=0.1396
k=7, R² score=0.1784
k=9, R² score=0.1959
k=11, R² score=0.2057
k=13, R² score=0.2136
k=15, R² score=0.2196

 best k: 15
 best R^2 score: 0.2195878208832538
